In [1]:
import pandas as pd

In [2]:
# Read agents and states
agents = pd.read_pickle("../../../data/agent_df_base_res_national_load_adjusted.pkl").reset_index(drop=False)
states = pd.read_csv("../../../state_input_csvs/states.csv", header = None, names = ['state_abbr', 'state_name'])

# Join to state name
agents = agents.merge(states, on='state_abbr', how='left')

# Sort by number of agents per state
sorted = agents.groupby(['state_abbr', 'state_name'], as_index = False).agg(agent_count=('agent_id', 'count')).sort_values(by='agent_count', ascending=False)

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/agent_df_base_res_national_load_adjusted.pkl'

In [ ]:
# Define state categories by agent count
large_states = sorted[sorted['agent_count'] > 1000]
mid_large_states = sorted[(sorted['agent_count'] > 500) & (sorted['agent_count'] <= 1000)]
mid_states = sorted[(sorted['agent_count'] > 100) & (sorted['agent_count'] <= 500)]
small_states = sorted[sorted['agent_count'] <= 100]

# Write states to CSV
large_states[['state_abbr', 'state_name']].to_csv("../../../state_input_csvs/large_states.csv", index=False, header=False)
mid_large_states[['state_abbr', 'state_name']].to_csv("../../../state_input_csvs/mid_large_states.csv", index=False, header=False)
mid_states[['state_abbr', 'state_name']].to_csv("../../../state_input_csvs/mid_states.csv", index=False, header=False)
small_states[['state_abbr', 'state_name']].to_csv("../../../state_input_csvs/small_states.csv", index=False, header=False)

# Write test CSVs
large_states[['state_abbr', 'state_name']].sample(n=4, random_state=42).to_csv("../../../state_input_csvs/large_states_test.csv", index=False, header=False)
mid_large_states[['state_abbr', 'state_name']].sample(n=12, random_state=42).to_csv("../../../state_input_csvs/mid_large_states_test.csv", index=False, header=False)
mid_states[['state_abbr', 'state_name']].sample(n=27, random_state=42).to_csv("../../../state_input_csvs/mid_states_test.csv", index=False, header=False)
small_states[['state_abbr', 'state_name']].sample(n=5, random_state=42).to_csv("../../../state_input_csvs/small_states_test.csv", index=False, header=False)

# Overall states
sorted[['state_abbr', 'state_name']].to_csv("../../../state_input_csvs/states.csv", index=False, header=False)
sorted[['state_abbr', 'state_name']].sample(n=10, random_state=42).to_csv("../../../state_input_csvs/states_test.csv", index=False, header=False)


In [ ]:
# Upload to GCE
!gsutil cp ../../../state_input_csvs/large_states.csv gs://dgen-assets/large_states.csv
!gsutil cp ../../../state_input_csvs/mid_large_states.csv gs://dgen-assets/mid_large_states.csv
!gsutil cp ../../../state_input_csvs/mid_states.csv gs://dgen-assets/mid_states.csv
!gsutil cp ../../../state_input_csvs/small_states.csv gs://dgen-assets/small_states.csv

!gsutil cp ../../../state_input_csvs/large_states_test.csv gs://dgen-assets/large_states_test.csv
!gsutil cp ../../../state_input_csvs/mid_large_states_test.csv gs://dgen-assets/mid_large_states_test.csv
!gsutil cp ../../../state_input_csvs/mid_states_test.csv gs://dgen-assets/mid_states_test.csv
!gsutil cp ../../../state_input_csvs/small_states_test.csv gs://dgen-assets/small_states_test.csv

!gsutil cp ../../../state_input_csvs/states.csv gs://dgen-assets/states.csv
!gsutil cp ../../../state_input_csvs/states_test.csv gs://dgen-assets/states_test.csv


In [2]:
# --- Verify the state CSVs the Cloud Batch jobs actually fetch -------------
# Derived from batch_job_yamls/ itself, so this list can't drift out of sync
# with the yamls again. Also checks each CSV's row count against that job's
# taskCount: a mismatch means some task index reads a blank line and fails.
import re
import subprocess
from pathlib import Path

REPO = Path("../../..").resolve()
yaml_dir = REPO / "batch_job_yamls"
csv_dir = REPO / "state_input_csvs"

yamls = list(yaml_dir.glob("*.yaml"))
yamls.sort()

manifest = {}  # csv filename -> list of (yaml filename, taskCount)
for y in yamls:
    text = y.read_text()
    m_csv = re.search(r"fetch_files\.py\s+dgen-assets\s+(\S+\.csv)", text)
    m_cnt = re.search(r"taskCount:\s*\"?(\d+)\"?", text)
    if m_csv:
        # the same CSV can be shared by several jobs (e.g. nj_state.csv)
        manifest.setdefault(m_csv.group(1), []).append(
            (y.name, int(m_cnt.group(1)) if m_cnt else None)
        )

names = list(manifest.keys())
names.sort()
print("{} state CSVs referenced by {} batch yamls\n".format(len(names), len(yamls)))

ok = True
for csv_name in names:
    jobs = manifest[csv_name]
    users = ", ".join(j[0] for j in jobs)
    p = csv_dir / csv_name
    if not p.exists():
        print("MISSING LOCALLY   {:32s} needed by {}".format(csv_name, users))
        ok = False
        continue
    n_rows = len([ln for ln in p.read_text().splitlines() if ln.strip()])
    bad = ["{}(taskCount={})".format(j[0], j[1]) for j in jobs if j[1] not in (None, n_rows)]
    flag = "   <-- MISMATCH " + ", ".join(bad) if bad else ""
    if bad:
        ok = False
    print("{:32s} rows={:3d}   {}{}".format(csv_name, n_rows, users, flag))

print("\nAll consistent - safe to upload." if ok else "\nFix the problems above BEFORE uploading.")


14 state CSVs referenced by 15 batch yamls

ca_state.csv                     rows=  1   dgen-batch-job-ca.yaml
large_states_r2.csv              rows=  2   dgen-batch-job-large-states-r2.yaml
large_states_test.csv            rows=  2   dgen-batch-job-large-states.yaml
mid_large_states_r2a.csv         rows=  3   dgen-batch-job-mid-large-states-r2a.yaml
mid_large_states_r2b.csv         rows=  4   dgen-batch-job-mid-large-states-r2b.yaml
mid_large_states_test.csv        rows=  3   dgen-batch-job-mid-large-states.yaml
mid_states_r2a.csv               rows=  6   dgen-batch-job-mid-states-r2a.yaml
mid_states_r2b.csv               rows=  7   dgen-batch-job-mid-states-r2b.yaml
mid_states_test.csv              rows=  3   dgen-batch-job-mid-states.yaml
nj_state.csv                     rows=  1   dgen-batch-job-nj-control.yaml, dgen-batch-job-nj-srec.yaml
small_states_r2a.csv             rows=  9   dgen-batch-job-small-states-r2a.yaml
small_states_r2b.csv             rows=  9   dgen-batch-job-smal

In [ ]:
# --- Upload them to the bucket ROOT (where fetch_files.py looks) -----------
# Run the verification cell above first; only run this if it reported clean.
# gsutil cp overwrites in place, so no need to delete the old objects.
for csv_name in names:
    p = csv_dir / csv_name
    if not p.exists():
        continue
    print("-> gs://dgen-assets/{}".format(csv_name))
    subprocess.run(["gsutil", "cp", str(p), "gs://dgen-assets/{}".format(csv_name)], check=True)

print("\nDone. Spot-check with:  !gsutil cat gs://dgen-assets/small_states_r2b.csv")
